# Kalibracja generatora i filtra

Cel: wygenerować ~50 obrazów SD 1.5 dla 1-2 ras, porównać CLIP/DINOv2 similarity do realnych refs, dobrać progi filtra

**Przed uruchomieniem:** Runtime -> Change runtime type -> **T4 GPU**

## 1. Klonowanie repo

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls

## 2. Instalacja zależności

In [ ]:
!pip install -q diffusers==0.31.0 accelerate==1.1.1 transformers==4.46.3 timm==1.0.11 PyYAML==6.0.2

## 3. Setup: device, imports, stałe

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from src.synth.filter import FilterConfig, compute_embeddings, nearest_sim
from src.synth.generate import GenerateConfig, build_pipeline, generate_for_breed
from src.synth.prompts import iter_prompt_variations
from src.utils.device import device_info, get_device
from src.utils.logger import CSVLogger

device = get_device()
print('device:', device_info(device))

BREEDS_TO_CALIBRATE = ['Ragdoll', 'Maine_Coon']
N_PER_BREED = 50
OUT_DIR = Path('data/synthetic/_calibration')
SPLIT_PATH = Path('data/splits/pets_10cls_30perclass_seed0.json')
IMAGES_DIR = Path('data/raw/oxford-iiit-pet/images')

## 4. Generacja - SD 1.5

In [ ]:
cfg = GenerateConfig(image_size=512, num_inference_steps=30, guidance_scale=7.5, batch_size=4)
pipe = build_pipeline(cfg, device)
print('pipeline ready')

In [ ]:
logger = CSVLogger(OUT_DIR, run_name='calibration')
generated_paths: dict[str, list[Path]] = {}

for breed in BREEDS_TO_CALIBRATE:
    pool = iter_prompt_variations(breed, mode='simple')
    prompts = [pool[i % len(pool)] for i in range(N_PER_BREED)]
    seeds = list(range(N_PER_BREED))
    out = OUT_DIR / breed
    print(f'[{breed}] generating {N_PER_BREED} -> {out}')
    paths = generate_for_breed(pipe, breed, prompts, seeds, out, cfg, device, logger)
    generated_paths[breed] = paths
    print(f'[{breed}] done: {len(paths)} images')

## 5. Sanity check: grid wygenerowanych obrazów

In [ ]:
def show_grid(paths, n_cols=6, title=''):
    paths = list(paths)[:n_cols * 3]
    n_rows = (len(paths) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 2.2))
    axes = np.array(axes).reshape(n_rows, n_cols)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, p in zip(axes.ravel(), paths):
        ax.imshow(Image.open(p))
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

for breed in BREEDS_TO_CALIBRATE:
    show_grid(generated_paths[breed], title=f'{breed} - synthetic')

## 6. Real refs ze splita (train, seed0)

30 obrazów/klasę

In [ ]:
with open(SPLIT_PATH) as f:
    split = json.load(f)
class_to_idx = split['meta']['class_to_idx']

real_paths: dict[str, list[Path]] = {}
for breed in BREEDS_TO_CALIBRATE:
    label = class_to_idx[breed]
    paths = [IMAGES_DIR / e['image'] for e in split['train'] if e['label'] == label]
    real_paths[breed] = paths
    print(f'{breed}: {len(paths)} real train images')

for breed in BREEDS_TO_CALIBRATE:
    show_grid(real_paths[breed], title=f'{breed} - real (train)')

## 7. Embeddingi: CLIP + DINOv2

Liczymy embeddingi dla:
- synth (50/rasa) - do oceny
- real train (30/rasa) - refs do similarity

Real-to-real (leave-one-out top-K) daje baseline "jak wygląda in-distribution similarity".

In [ ]:
fc = FilterConfig()

embeddings = {}
for breed in BREEDS_TO_CALIBRATE:
    for kind, model_id in [('clip', fc.clip_model_id), ('dinov2', fc.dinov2_model_id)]:
        for tag, paths in [('synth', generated_paths[breed]), ('real', real_paths[breed])]:
            key = (breed, kind, tag)
            embeddings[key] = compute_embeddings(paths, kind, model_id, device, batch_size=16)
            print(f'{key}: {embeddings[key].shape}')

In [ ]:
def real_to_real_loo(emb: torch.Tensor, top_k: int = 5) -> torch.Tensor:
    sim = emb @ emb.T
    sim.fill_diagonal_(-1.0)
    k = min(top_k, emb.shape[0] - 1)
    topk, _ = sim.topk(k=k, dim=1)
    return topk.mean(dim=1)

scores = {}
for breed in BREEDS_TO_CALIBRATE:
    for kind in ('clip', 'dinov2'):
        real = embeddings[(breed, kind, 'real')]
        synth = embeddings[(breed, kind, 'synth')]
        scores[(breed, kind, 'synth')] = nearest_sim(synth, real, top_k=fc.top_k).numpy()
        scores[(breed, kind, 'real')] = real_to_real_loo(real, top_k=fc.top_k).numpy()

## 8. Histogramy: synth vs real (in-distribution baseline)

In [ ]:
fig, axes = plt.subplots(len(BREEDS_TO_CALIBRATE), 2, figsize=(11, 3.5 * len(BREEDS_TO_CALIBRATE)))
axes = np.array(axes).reshape(len(BREEDS_TO_CALIBRATE), 2)
for r, breed in enumerate(BREEDS_TO_CALIBRATE):
    for c, kind in enumerate(['clip', 'dinov2']):
        ax = axes[r, c]
        ax.hist(scores[(breed, kind, 'real')], bins=15, alpha=0.6, label='real-to-real (LOO top-K)', color='tab:green')
        ax.hist(scores[(breed, kind, 'synth')], bins=15, alpha=0.6, label='synth-to-real top-K', color='tab:orange')
        ax.set_title(f'{breed} - {kind}')
        ax.set_xlabel('cosine sim'); ax.set_ylabel('count'); ax.legend()
plt.tight_layout(); plt.show()

## 9. Sweep progów - ile syntetyków zostaje?

Dla każdej rasy sprawdzamy ile obrazów przejdzie filtr przy różnych wartościach `floor_clip` i `floor_dinov2`. Cel: zostawić ~60-75% syntetyków

In [ ]:
import pandas as pd

rows = []
for breed in BREEDS_TO_CALIBRATE:
    c = scores[(breed, 'clip', 'synth')]
    d = scores[(breed, 'dinov2', 'synth')]
    for fc_floor in [0.55, 0.60, 0.65, 0.70]:
        for fd_floor in [0.45, 0.50, 0.55, 0.60]:
            kept = ((c >= fc_floor) & (d >= fd_floor)).sum()
            rows.append({
                'breed': breed,
                'floor_clip': fc_floor,
                'floor_dinov2': fd_floor,
                'kept': int(kept),
                'kept_pct': round(100 * kept / len(c), 1),
            })
df = pd.DataFrame(rows)
df_pivot = df.pivot_table(index=['breed', 'floor_clip'], columns='floor_dinov2', values='kept_pct')
df_pivot

## 10. Decyzja

In [ ]:
PROPOSED_FLOOR_CLIP = 0.65
PROPOSED_FLOOR_DINOV2 = 0.55

for breed in BREEDS_TO_CALIBRATE:
    c = scores[(breed, 'clip', 'synth')]
    d = scores[(breed, 'dinov2', 'synth')]
    mask_kept = (c >= PROPOSED_FLOOR_CLIP) & (d >= PROPOSED_FLOOR_DINOV2)
    paths = generated_paths[breed]
    kept = [p for p, m in zip(paths, mask_kept) if m]
    dropped = [p for p, m in zip(paths, mask_kept) if not m]
    print(f'{breed}: kept={len(kept)}/{len(paths)} dropped={len(dropped)}')
    show_grid(kept[:12], title=f'{breed} - KEPT (top 12)')
    if dropped:
        show_grid(dropped[:12], title=f'{breed} - DROPPED (top 12)')